# Submission · Downscaling GFS 27 km → WRF 3 km (Cordillera Blanca)

**Equipo Novatech · Hackathon IA PUCP 2026**

Ensamble de tres familias de modelos, todos entrenados **solo con la data oficial** de la competencia:

1. **Ridge** sobre la ventana GFS completa (t-2, t-1, t) + hora del día.
2. **Red convolucional simple**: encoder en la grilla GFS → `grid_sample` a la grilla WRF → decoder con relieve.
3. **U-Net residual**: igual, pero con bloques residuales y un nivel de menor resolución.

Todas predicen la **anomalía respecto a la climatología por hora** y usan una pérdida ponderada por variable
para que coincida con la métrica (1 − media de RMSE/RMSE_climatología).

Validación: corte temporal (desde el 16 de junio) con purga de 48 h. Nunca split aleatorio.
Con `VALIDAR = True` se reporta el score local antes de entrenar con todo el train.

In [ ]:
import glob, os, time
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.linear_model import Ridge

VALIDAR = False          # True: entrena primero con el holdout y muestra el score local
SEMILLAS_SIMPLE = [11, 12, 13]
SEMILLAS_UNET = [21, 22]
PESO_RIDGE = 0.35
QUICK = os.environ.get('QUICK') == '1'   # solo para probar que todo corre
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEV)

def find(name):
    for p in [f'/kaggle/input/**/{name}', f'../data/**/{name}', f'data/**/{name}', f'**/{name}']:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    raise FileNotFoundError(name)

tr, te, st = np.load(find('train.npz')), np.load(find('test.npz')), dict(np.load(find('static.npz')))
Xtr, Ytr, ttr = tr['X'], tr['Y'], tr['times'].astype(str)
Xte, tte = te['X'], te['times'].astype(str)
VARS = ['RH', 'T', 'U', 'V']
EVAL_HOURS = list(range(0, 24, 3))
print(Xtr.shape, Ytr.shape, Xte.shape)

## Utilidades: ventanas temporales, climatología, métrica y corte de validación

In [ ]:
def hours_of(t):
    return np.array([int(str(k)[-2:]) for k in t])

def make_windows(X, t, lags=(2, 1, 0)):
    """Ventana [t-2, t-1, t] por tiempo real; si falta una hora previa se usa la hora t."""
    ts = pd.to_datetime(np.asarray(t), format='%Y%m%d%H')
    pos = {k: i for i, k in enumerate(ts)}
    out = np.empty((len(X), len(lags)) + X.shape[1:], np.float32)
    for i, k in enumerate(ts):
        for j, lag in enumerate(lags):
            idx = pos.get(k - pd.Timedelta(hours=lag))
            out[i, j] = X[idx] if idx is not None else X[i]
    return out

def climatology(Y, t):
    h = hours_of(t)
    return np.stack([Y[h == k].mean(0) for k in range(24)])

def skill_score(pred, true, t, clim):
    ref = clim[hours_of(t)]
    r_m = np.sqrt(((pred.astype(np.float64) - true) ** 2).mean((0, 1, 2)))
    r_c = np.sqrt(((ref.astype(np.float64) - true) ** 2).mean((0, 1, 2)))
    ratio = r_m / r_c
    return float(1 - ratio.mean()), dict(zip(VARS, np.round(1 - ratio, 4)))

def temporal_holdout(t, val_start='2024061600', purge_hours=48):
    ts = pd.to_datetime(np.asarray(t), format='%Y%m%d%H')
    cut = pd.to_datetime(val_start, format='%Y%m%d%H')
    return np.asarray(ts < cut - pd.Timedelta(hours=purge_hours)), np.asarray(ts >= cut)

Wtr, Wte = make_windows(Xtr, ttr), make_windows(Xte, tte)
h_tr, h_te = hours_of(ttr), hours_of(tte)

## Modelo 1 · Ridge sobre la ventana GFS

In [ ]:
def ridge_predict(trn, W_pred, h_pred, alpha=1500):
    clim = climatology(Ytr[trn], ttr[trn])
    feats = lambda W, h: np.concatenate([W.reshape(len(W), -1), np.eye(24)[h]], 1)
    Ftr = feats(Wtr[trn], h_tr[trn]); mu, sd = Ftr.mean(0), Ftr.std(0) + 1e-6
    A = (Ytr[trn] - clim[h_tr[trn]]).reshape(trn.sum(), -1)
    m = Ridge(alpha=alpha).fit((Ftr - mu) / sd, A)
    P = m.predict((feats(W_pred, h_pred) - mu) / sd).reshape(-1, 70, 35, 4)
    return (P + clim[h_pred]).astype(np.float32)

## Modelos 2 y 3 · Redes convolucionales GFS → WRF

In [ ]:
glat, glon, wlat, wlon = st['gfs_lats'], st['gfs_lons'], st['wrf_lats'], st['wrf_lons']
LAT, LON = np.meshgrid(wlat, wlon, indexing='ij')
gy = (LAT - glat[0]) / (glat[-1] - glat[0]) * 2 - 1      # GFS: fila 0 = norte
gx = (LON - glon[0]) / (glon[-1] - glon[0]) * 2 - 1
GRID = torch.tensor(np.stack([gx, gy], -1)[None], dtype=torch.float32, device=DEV)
dem = st['dem']; dy, dx = np.gradient(dem)
STAT = torch.tensor(np.stack([(dem - dem.mean()) / dem.std(), dy / dy.std(), dx / dx.std(),
                              (LAT - LAT.mean()) / LAT.std(), (LON - LON.mean()) / LON.std()])[None],
                    dtype=torch.float32, device=DEV)   # relieve, pendientes, lat, lon

def entrada_gfs(x, h):
    B = x.shape[0]
    x = x.permute(0, 1, 4, 2, 3).reshape(B, 12, 17, 13)
    ang = 2 * np.pi * h / 24
    hh = torch.stack([torch.sin(ang), torch.cos(ang)], 1)[..., None, None].expand(B, 2, 17, 13)
    return torch.cat([x, hh], 1)

class Simple(nn.Module):
    def __init__(self, w=64):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv2d(14, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, w, 3, padding=1), nn.GELU())
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35))
        self.dec = nn.Sequential(nn.Conv2d(w + 13, w, 3, padding=1), nn.GELU(),
                                 nn.Conv2d(w, w, 3, padding=1), nn.GELU(), nn.Conv2d(w, 4, 1))
    def forward(self, x, h):
        B = x.shape[0]
        f = F.grid_sample(self.enc(entrada_gfs(x, h)), GRID.expand(B, -1, -1, -1), align_corners=True)
        f = torch.cat([f, STAT.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1)
        return self.dec(f).permute(0, 2, 3, 1)

class RB(nn.Module):
    def __init__(self, c, d):
        super().__init__()
        self.c1, self.c2 = nn.Conv2d(c, c, 3, padding=1), nn.Conv2d(c, c, 3, padding=1)
        self.n1, self.n2, self.d = nn.GroupNorm(8, c), nn.GroupNorm(8, c), nn.Dropout2d(d)
    def forward(self, x):
        return x + self.c2(self.d(F.gelu(self.n2(self.c1(F.gelu(self.n1(x)))))))

class UNet(nn.Module):
    def __init__(self, w=64, nb=2, d=0.0):
        super().__init__()
        self.inp = nn.Conv2d(14, w, 3, padding=1); self.enc = nn.Sequential(*[RB(w, d) for _ in range(nb)])
        self.static = nn.Parameter(torch.zeros(1, 8, 70, 35)); self.mid = nn.Conv2d(w + 13, w, 3, padding=1)
        self.down = nn.Conv2d(w, w, 3, stride=2, padding=1); self.bott = nn.Sequential(*[RB(w, d) for _ in range(nb)])
        self.fuse = nn.Conv2d(2 * w, w, 1); self.dec = nn.Sequential(*[RB(w, d) for _ in range(nb)])
        self.out = nn.Conv2d(w, 4, 1)
    def forward(self, x, h):
        B = x.shape[0]
        f = F.grid_sample(self.enc(self.inp(entrada_gfs(x, h))), GRID.expand(B, -1, -1, -1), align_corners=True)
        f = self.mid(torch.cat([f, STAT.expand(B, -1, -1, -1), self.static.expand(B, -1, -1, -1)], 1))
        g = F.interpolate(self.bott(self.down(f)), size=(70, 35), mode='bilinear', align_corners=False)
        return self.out(F.gelu(self.dec(self.fuse(torch.cat([f, g], 1))))).permute(0, 2, 3, 1)

In [ ]:
def entrenar_red(trn, arch, seed, W_pred, h_pred, epochs=25, lr=2e-3, wd=1e-4, drop=0.0, val=None):
    torch.manual_seed(seed); np.random.seed(seed)
    if QUICK: epochs = 1
    T = lambda z: torch.tensor(z, dtype=torch.float32, device=DEV)
    clim = climatology(Ytr[trn], ttr[trn])
    mu_x, sd_x = Xtr[trn].mean((0, 1, 2)), Xtr[trn].std((0, 1, 2))
    anom = Ytr - clim[h_tr]; asd = anom[trn].std((0, 1, 2))
    crm = np.sqrt(((clim[h_tr[trn]] - Ytr[trn]) ** 2).mean((0, 1, 2)))
    wv = (asd / crm) ** 2; wv = T(wv / wv.mean())            # pérdida alineada con la métrica
    Wn, An, CL, AS = T((Wtr - mu_x) / sd_x), T(anom / asd), T(clim), T(asd)
    m = (Simple() if arch == 'simple' else UNet(d=drop)).to(DEV)
    ema = torch.optim.swa_utils.AveragedModel(m, multi_avg_fn=torch.optim.swa_utils.get_ema_multi_avg_fn(0.995))
    itr, BS = np.where(trn)[0], 32
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * int(np.ceil(len(itr) / BS)))
    for ep in range(epochs):
        m.train(); np.random.shuffle(itr)
        for s in range(0, len(itr), BS):
            b = itr[s:s + BS]; hb = torch.tensor(h_tr[b], device=DEV)
            loss = (((m(Wn[b], hb.float()) - An[b]) ** 2).mean((0, 1, 2)) * wv).mean()
            opt.zero_grad(); loss.backward(); opt.step(); sch.step(); ema.update_parameters(m)
    net = ema.module.eval(); Wp = T((W_pred - mu_x) / sd_x); out = []
    with torch.no_grad():
        for s in range(0, len(Wp), 256):
            hb = torch.tensor(h_pred[s:s + 256], device=DEV)
            out.append((CL[hb] + net(Wp[s:s + 256], hb.float()) * AS).cpu().numpy())
    return np.concatenate(out).astype(np.float32), net

def ensamble(trn, W_pred, h_pred):
    t0 = time.time()
    P_ridge = ridge_predict(trn, W_pred, h_pred)
    redes, nets = [], []
    for s in SEMILLAS_SIMPLE:
        p, n = entrenar_red(trn, 'simple', s, W_pred, h_pred); redes.append(p); nets.append(n)
        print(f'  simple seed {s} listo · {time.time() - t0:.0f}s')
    for i, s in enumerate(SEMILLAS_UNET):
        kw = dict(epochs=20) if i % 2 == 0 else dict(epochs=20, drop=0.1, wd=0.01)
        p, n = entrenar_red(trn, 'unet', s, W_pred, h_pred, **kw); redes.append(p); nets.append(n)
        print(f'  unet seed {s} listo · {time.time() - t0:.0f}s')
    return PESO_RIDGE * P_ridge + (1 - PESO_RIDGE) * np.mean(redes, 0), nets

## Validación temporal (opcional)

In [ ]:
if VALIDAR:
    trn, val = temporal_holdout(ttr)
    P_val, _ = ensamble(trn, Wtr[val], h_tr[val])
    ev = np.isin(h_tr[val], EVAL_HOURS)
    clim_trn = climatology(Ytr[trn], ttr[trn])
    print('Score local (holdout 16–30 jun):', skill_score(P_val[ev], Ytr[val][ev], ttr[val][ev], clim_trn))

## Entrenamiento final con todo el train y submission

In [ ]:
todo = np.ones(len(ttr), bool)
P_test, nets = ensamble(todo, Wte, h_te)

sample = pd.read_csv(find('sample_submission.csv'))
pos = {k: i for i, k in enumerate(tte)}
parts = sample['id'].str.split('_', expand=True)
ti, r, c = parts[0].map(pos).values, parts[1].astype(int).values, parts[2].astype(int).values
sub = pd.DataFrame({'id': sample['id']})
for k, v in enumerate(VARS):
    sub[v] = P_test[ti, r, c, k].round(3)
sub.to_csv('submission.csv', index=False)
print(sub.shape); sub.head()

In [ ]:
# Pesos de las redes (los mismos del submission) para reutilizarlos en el producto
os.makedirs('modelos', exist_ok=True)
for i, n in enumerate(nets):
    torch.save(n.state_dict(), f'modelos/red_{i}.pt')
print(sorted(os.listdir('modelos')))